# Subtype model evaluation

Compares the best subtype classifiers overall and by GEP group.

This notebook accompanies [Stigmatizing Language in Gender-Expansive Patient Records: Corpus Development, Disparity Analysis, and Natural Language Processing-Based Detection Study](https://www.jmir.org/2026/1/e91089).

## Data and execution requirements

- Clinical note text and MIMIC identifiers are not included in this repository.
- Run this notebook only in an environment authorized to access MIMIC-IV and the credentialed annotation release.
- Set `GEP_DATA_DIR`, `GEP_MODEL_DIR`, `GEP_RESULTS_DIR`, and `GEP_FIGURES_DIR` as needed. By default, repository-local directories are used.
- The notebook outputs and execution counters have been removed from the public version.


In [ ]:
# Repository-local path configuration
from pathlib import Path
import os

PROJECT_ROOT = Path(os.environ.get("GEP_PROJECT_ROOT", Path.cwd())).resolve()
DATA_DIR = Path(os.environ.get("GEP_DATA_DIR", PROJECT_ROOT / "data")).resolve()
MODEL_DIR = Path(os.environ.get("GEP_MODEL_DIR", PROJECT_ROOT / "models")).resolve()
RESULTS_DIR = Path(os.environ.get("GEP_RESULTS_DIR", PROJECT_ROOT / "results")).resolve()
FIGURES_DIR = Path(os.environ.get("GEP_FIGURES_DIR", PROJECT_ROOT / "figures")).resolve()

for directory in (MODEL_DIR, RESULTS_DIR, FIGURES_DIR):
    directory.mkdir(parents=True, exist_ok=True)


In [ ]:
import os

# ==== Configuration ====

# === Define paths ===
ACC_ML_DIR = str(MODEL_DIR / 'GEP_SUBTYPES_Accuracy')
F1_ML_DIR  = str(MODEL_DIR / 'GEP_SUBTYPES')

SUBTYPES = ['Credibility and Obstinacy', 'Compliance', 'Descriptors', 'Misgendering']

# Accuracy-optimized best models
ACC_ML_BEST = {
    'Compliance': {
        'LR':  'LR_tfidf_sw',
        'NB':  'NB_tfidf_sw',
        'RF':  'RF_count',
        'SVM': 'SVM_tfidf_sw',
    },
    'Credibility and Obstinacy': {
        'LR':  'LR_tfidf_sw',
        'NB':  'NB_tfidf',
        'RF':  'RF_count',
        'SVM': 'SVM_tfidf',
    },
    'Descriptors': {
        'LR':  'LR_tfidf',
        'NB':  'NB_tfidf',
        'RF':  'RF_count_sw',
        'SVM': 'SVM_tfidf_sw',
    },
    'Misgendering': {
        'LR':  'LR_count',
        'NB':  'NB_count_sw',
        'RF':  'RF_tfidf_sw',
        'SVM': 'SVM_count',
    },
}

# F1-optimized best models
F1_ML_BEST = {
    'Compliance': {
        'LR':  'LR_tfidf_sw',
        'NB':  'NB_tfidf_sw',
        'RF':  'RF_tfidf_sw',
        'SVM': 'SVM_tfidf_sw',
    },
    'Credibility and Obstinacy': {
        'LR':  'LR_count_sw',
        'NB':  'NB_count',
        'RF':  'RF_tfidf_sw',
        'SVM': 'SVM_tfidf',
    },
    'Descriptors': {
        'LR':  'LR_tfidf_sw',
        'NB':  'NB_count_sw',
        'RF':  'RF_count_sw',
        'SVM': 'SVM_tfidf_sw',
    },
    'Misgendering': {
        'LR':  'LR_count_sw',
        'NB':  'NB_tfidf',
        'RF':  'RF_tfidf',
        'SVM': 'SVM_count',
    },
}

def check_models(base_dir, winners, label):
    print(f"\n=== Checking {label} models in: {base_dir} ===")
    found, missing = 0, 0
    for subtype, fams in winners.items():
        subtype_tag = subtype.replace(" ", "_")
        for fam, model_name in fams.items():
            fname = f"{model_name}_{subtype_tag}_best_model.joblib"
            fpath = os.path.join(base_dir, fname)
            if os.path.exists(fpath):
                print(f" {fname}")
                found += 1
            else:
                print(f" {fname}  (missing)")
                missing += 1
    print(f"\nSummary for {label}: {found} found, {missing} missing.\n")

# Run checks
check_models(ACC_ML_DIR, ACC_ML_BEST, "Accuracy-optimized Traditional ML")
check_models(F1_ML_DIR, F1_ML_BEST, "F1-optimized Traditional ML")


In [ ]:
# === 5.3.7 Subtype evaluation: best models (Accuracy- and F1-optimized) ===
import os, joblib, numpy as np, pandas as pd, torch, math
from tqdm import tqdm
from typing import Dict, List
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    LongformerTokenizer, LongformerForSequenceClassification
)

# -------------------------
# Paths & constants (from you)
# -------------------------
TEST_CSV = str(DATA_DIR / 'GEP_test_80_20.csv')
RESULTS_DIR = str(RESULTS_DIR) + os.sep
os.makedirs(RESULTS_DIR, exist_ok=True)

# Traditional ML
ACC_ML_DIR = str(MODEL_DIR / 'GEP_SUBTYPES_Accuracy')
F1_ML_DIR  = str(MODEL_DIR / 'GEP_SUBTYPES')

# Transformers (HF dirs grouped by metric target)
TXF_DIRS = {
    "Accuracy": {
        "BERT":        str(MODEL_DIR / 'BERT_GEP_SUBTYPES'),
        "ClinicalBERT":str(MODEL_DIR / 'clinicalbert_GEP_SUBTYPES'),
        "Longformer":  str(MODEL_DIR / 'Longformer_GEP_SUBTYPES'),
    },
    "F1": {
        "BERT":        str(MODEL_DIR / 'BERT_GEP_SUBTYPES_F1'),
        "ClinicalBERT":str(MODEL_DIR / 'clinicalbert_GEP_SUBTYPES_F1'),
        "Longformer":  str(MODEL_DIR / 'Longformer_GEP_SUBTYPES_F1'),
    }
}

SUBTYPES = ['Credibility and Obstinacy', 'Compliance', 'Descriptors', 'Misgendering']

# -------------------------
# Best traditional ML models you provided (Model column names) per subtype
# NOTE: We assume your saving pattern:
#   f"{Model}_{subtype.replace(' ', '_')}_best_model.joblib"
# and that these files live under the base dir (ACC_ML_DIR or F1_ML_DIR).
# -------------------------

# Accuracy-optimized winners
ACC_ML_BEST = {
    'Compliance': {
        'LR':  'LR_tfidf_sw',
        'NB':  'NB_tfidf_sw',
        'RF':  'RF_count',
        'SVM': 'SVM_tfidf_sw',
    },
    'Credibility and Obstinacy': {
        'LR':  'LR_tfidf_sw',
        'NB':  'NB_tfidf',
        'RF':  'RF_count',
        'SVM': 'SVM_tfidf',
    },
    'Descriptors': {
        'LR':  'LR_tfidf',
        'NB':  'NB_tfidf',
        'RF':  'RF_count_sw',
        'SVM': 'SVM_tfidf_sw',
    },
    'Misgendering': {
        'LR':  'LR_count',
        'NB':  'NB_count_sw',
        'RF':  'RF_tfidf_sw',
        'SVM': 'SVM_count',
    },
}

# F1-optimized winners
F1_ML_BEST = {
    'Compliance': {
        'LR':  'LR_tfidf_sw',
        'NB':  'NB_tfidf_sw',
        'RF':  'RF_tfidf_sw',
        'SVM': 'SVM_tfidf_sw',
    },
    'Credibility and Obstinacy': {
        'LR':  'LR_count_sw',
        'NB':  'NB_count',
        'RF':  'RF_tfidf_sw',
        'SVM': 'SVM_tfidf',
    },
    'Descriptors': {
        'LR':  'LR_tfidf_sw',
        'NB':  'NB_count_sw',
        'RF':  'RF_count_sw',
        'SVM': 'SVM_tfidf_sw',
    },
    'Misgendering': {
        'LR':  'LR_count_sw',
        'NB':  'NB_tfidf',
        'RF':  'RF_tfidf',
        'SVM': 'SVM_count',
    },
}

# -------------------------
# Data
# -------------------------
df = pd.read_csv(TEST_CSV)
texts = df['text'].astype(str).tolist()
# For each subtype we'll pick labels dynamically
gep = df['GEP'].astype(int).to_numpy()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------
# Metrics helper
# -------------------------
def compute_metrics(y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    if y_true.min() == y_true.max():
        auc = float("nan")
    else:
        try:
            auc = roc_auc_score(y_true, y_prob)
        except Exception:
            auc = float("nan")
    return {"Accuracy": acc, "Precision": prec, "Recall": rec, "F1": f1, "AUC": auc}

# -------------------------
# Traditional ML evaluation
# -------------------------
def eval_traditional(model_path: str, X: List[str], y: np.ndarray, group_mask: np.ndarray = None):
    model = joblib.load(model_path)

    def _predict(xx):
        y_pred = model.predict(xx)
        if hasattr(model, "predict_proba"):
            y_prob = model.predict_proba(xx)[:, 1]
        elif hasattr(model, "decision_function"):
            y_prob = model.decision_function(xx)
        else:
            # best effort: cast prediction to float
            y_prob = y_pred.astype(float)
        return y_pred, y_prob

    # overall
    y_pred, y_prob = _predict(X)
    overall = compute_metrics(y, y_pred, y_prob)

    # groups
    out = {"Overall": overall}
    if group_mask is not None:
        for gval, gname in [(0, "GEP=0"), (1, "GEP=1")]:
            idx = np.where(group_mask == gval)[0]
            if idx.size == 0:
                out[gname] = {k: float("nan") for k in overall.keys()}
                out[gname]["N"] = 0
            else:
                yp, ypp = _predict([X[i] for i in idx])
                out[gname] = compute_metrics(y[idx], yp, ypp)
                out[gname]["N"] = int(idx.size)
    return out

# -------------------------
# Transformer datasets & evaluation
# (BERT/ClinicalBERT: chunking+max pooling; Longformer: full 4096)
# -------------------------
class ChunkedTextDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, chunk_size=510, max_length=512, doc_max_len=4096):
        self.texts = texts
        self.labels = labels
        self.tok = tokenizer
        self.chunk_size = chunk_size
        self.max_length = max_length
        self.doc_max_len = doc_max_len

    def __len__(self): return len(self.texts)

    def _chunk_ids(self, text):
        ids = self.tok.encode(text, add_special_tokens=False, truncation=False)[: self.doc_max_len]
        chunks = []
        for i in range(0, len(ids), self.chunk_size):
            core = ids[i:i+self.chunk_size]
            ch = [self.tok.cls_token_id] + core + [self.tok.sep_token_id]
            if len(ch) < self.max_length:
                ch += [self.tok.pad_token_id] * (self.max_length - len(ch))
            else:
                ch = ch[:self.max_length]
            chunks.append(ch)
        if not chunks:
            ch = [self.tok.cls_token_id, self.tok.sep_token_id]
            ch += [self.tok.pad_token_id] * (self.max_length - len(ch))
            chunks = [ch]
        return chunks

    def __getitem__(self, idx):
        chunks = self._chunk_ids(self.texts[idx])
        return {
            "chunks": torch.tensor(chunks, dtype=torch.long),
            "label": float(self.labels[idx]),
            "num_chunks": len(chunks)
        }

def collate_chunks(batch, pad_id):
    flat = torch.cat([b["chunks"] for b in batch], dim=0)
    labels = torch.tensor([b["label"] for b in batch], dtype=torch.float)
    num_chunks = [b["num_chunks"] for b in batch]
    return {"chunks": flat, "labels": labels, "num_chunks": num_chunks, "pad_id": pad_id}

class LongTextDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=4096, use_global=False):
        self.texts = texts
        self.labels = labels
        self.tok = tokenizer
        self.max_length = max_length
        self.use_global = use_global

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tok(self.texts[idx], return_tensors="pt", truncation=True, padding="max_length", max_length=self.max_length)
        input_ids = enc["input_ids"].squeeze(0)
        attn = enc["attention_mask"].squeeze(0)
        out = {"input_ids": input_ids, "attention_mask": attn, "label": float(self.labels[idx])}
        if self.use_global:
            gam = torch.zeros_like(input_ids); gam[0] = 1
            out["global_attention_mask"] = gam
        return out

def collate_longformer(batch):
    input_ids = torch.stack([b["input_ids"] for b in batch])
    attention_mask = torch.stack([b["attention_mask"] for b in batch])
    labels = torch.tensor([b["label"] for b in batch], dtype=torch.float)
    out = {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}
    if "global_attention_mask" in batch[0]:
        out["global_attention_mask"] = torch.stack([b["global_attention_mask"] for b in batch])
    return out

@torch.no_grad()
def eval_transformer_dir(model_dir, family, texts, labels, group_mask=None, batch_size=48):
    """
    family: 'BERT', 'ClinicalBERT', or 'Longformer'
    Returns dict: {'Overall': {...}, 'GEP=0': {...}, 'GEP=1': {...}}
    """
    if family == "Longformer":
        tok = LongformerTokenizer.from_pretrained(model_dir)
        mdl = LongformerForSequenceClassification.from_pretrained(model_dir).to(device).eval()
        ds = LongTextDataset(texts, labels, tok, max_length=4096, use_global=False)
        dl = torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=False, collate_fn=collate_longformer)
        logits_all, y_all = [], []
        for batch in tqdm(dl, desc=f"Eval {family}"):
            input_ids = batch["input_ids"].to(device)
            attn = batch["attention_mask"].to(device)
            gam = batch.get("global_attention_mask")
            gam = gam.to(device) if gam is not None else None
            out = mdl(input_ids=input_ids, attention_mask=attn, global_attention_mask=gam)
            logits_all.append(out.logits.view(-1).cpu().numpy())
            y_all.append(batch["labels"].cpu().numpy())
        logits = np.concatenate(logits_all); y_true = np.concatenate(y_all).astype(int)
        probs = 1/(1+np.exp(-logits)); preds = (probs>=0.5).astype(int)

        results = {"Overall": compute_metrics(y_true, preds, probs)}
        if group_mask is not None:
            for gval, gname in [(0,"GEP=0"),(1,"GEP=1")]:
                idx = np.where(group_mask==gval)[0]
                if idx.size == 0:
                    r = {k: float("nan") for k in results["Overall"].keys()}
                    r["N"] = 0
                else:
                    r = compute_metrics(y_true[idx], preds[idx], probs[idx])
                    r["N"] = int(idx.size)
                results[gname] = r
        return results

    else:
        # BERT / ClinicalBERT -> chunking + max-logit pooling
        tok = AutoTokenizer.from_pretrained(model_dir)
        if tok.pad_token_id is None:
            tok.pad_token = tok.sep_token
        mdl = AutoModelForSequenceClassification.from_pretrained(model_dir).to(device).eval()
        ds = ChunkedTextDataset(texts, labels, tok, chunk_size=510, max_length=512, doc_max_len=4096)
        def _collate(b): return collate_chunks(b, tok.pad_token_id)
        dl = torch.utils.data.DataLoader(ds, batch_size=48, shuffle=False, collate_fn=_collate)

        pooled_logits, y_true = [], []
        for batch in tqdm(dl, desc=f"Eval {family}"):
            chunks = batch["chunks"].to(device)
            pad_id = batch["pad_id"]
            num_chunks = batch["num_chunks"]
            out = mdl(input_ids=chunks, attention_mask=(chunks != pad_id))
            chunk_logits = out.logits.view(-1)

            start = 0
            for nc, y in zip(num_chunks, batch["labels"]):
                doc_logits = chunk_logits[start:start+nc]
                pooled = torch.max(doc_logits)
                pooled_logits.append(pooled.item())
                y_true.append(int(y.item()))
                start += nc

        logits = np.array(pooled_logits); probs = 1/(1+np.exp(-logits))
        preds = (probs>=0.5).astype(int); y_true = np.array(y_true)

        results = {"Overall": compute_metrics(y_true, preds, probs)}
        if group_mask is not None:
            for gval, gname in [(0,"GEP=0"),(1,"GEP=1")]:
                idx = np.where(group_mask==gval)[0]
                if idx.size == 0:
                    r = {k: float("nan") for k in results["Overall"].keys()}
                    r["N"] = 0
                else:
                    r = compute_metrics(y_true[idx], preds[idx], probs[idx])
                    r["N"] = int(idx.size)
                results[gname] = r
        return results

# -------------------------
# Driver: build the experiment list and evaluate
# -------------------------
all_rows = []

# 1) Traditional ML (Accuracy-optimized & F1-optimized)
for optim_name, base_dir, winners in [
    ("Accuracy-optimized", ACC_ML_DIR, ACC_ML_BEST),
    ("F1-optimized",       F1_ML_DIR,  F1_ML_BEST),
]:
    for subtype in SUBTYPES:
        labels = df[subtype].astype(int).to_numpy()
        subtype_tag = subtype.replace(" ", "_")
        for fam in ["LR","NB","RF","SVM"]:
            model_short = winners[subtype][fam]  # e.g., "LR_tfidf_sw"
            fname = f"{model_short}_{subtype_tag}_best_model.joblib"
            mpath = os.path.join(base_dir, fname)
            if not os.path.exists(mpath):
                print(f"[WARN] Missing traditional model: {mpath}")
                continue
            res = eval_traditional(mpath, texts, labels, group_mask=gep)
            for group_name, mets in res.items():
                row = {
                    "Optimization": optim_name,
                    "Family": f"Traditional-{fam}",
                    "ModelId": model_short,
                    "Subtype": subtype,
                    "Group": group_name,
                    "N": (int(len(labels)) if group_name=="Overall" else mets.get("N", int(np.sum(gep==int(group_name[-1])))) ),
                    "Accuracy": mets["Accuracy"],
                    "Precision": mets["Precision"],
                    "Recall": mets["Recall"],
                    "F1": mets["F1"],
                    "AUC": mets["AUC"],
                }
                all_rows.append(row)

# 2) Transformers (Accuracy-optimized & F1-optimized)
for optim_name, fam_dirs in TXF_DIRS.items():
    for family, base_dir in fam_dirs.items():
        for subtype in SUBTYPES:
            labels = df[subtype].astype(int).to_numpy()
            mdir = os.path.join(base_dir, subtype.replace(" ", "_"))
            if not os.path.isdir(mdir):
                print(f"[WARN] Missing transformer dir: {mdir}")
                continue
            res = eval_transformer_dir(mdir, family, texts, labels, group_mask=gep, batch_size=48 if family!="Longformer" else 48)
            for group_name, mets in res.items():
                row = {
                    "Optimization": optim_name,
                    "Family": family,
                    "ModelId": os.path.basename(mdir),
                    "Subtype": subtype,
                    "Group": group_name,
                    "N": (int(len(labels)) if group_name=="Overall" else mets.get("N", int(np.sum(gep==int(group_name[-1])))) ),
                    "Accuracy": mets["Accuracy"],
                    "Precision": mets["Precision"],
                    "Recall": mets["Recall"],
                    "F1": mets["F1"],
                    "AUC": mets["AUC"],
                }
                all_rows.append(row)

# -------------------------
# Save & quick preview
# -------------------------
out_df = pd.DataFrame(all_rows)
out_csv = os.path.join(RESULTS_DIR, "subtype_best_models_overall_and_GEP_split.csv")
out_xlsx = os.path.join(RESULTS_DIR, "subtype_best_models_overall_and_GEP_split.xlsx")
out_df.to_csv(out_csv, index=False)
try:
    out_df.to_excel(out_xlsx, index=False)
except Exception:
    pass

print("\n Evaluation complete.")
print(f"Saved:\n - {out_csv}\n - {out_xlsx}")
print("\nHead:")
print(out_df.head(12))


In [ ]:
out_df.shape

In [ ]:
# === Evaluate MSTL Longformer models (exact subtype mapping) ===
MSTL_LONGFORMER_DIRS = {
    'Credibility and Obstinacy': str(MODEL_DIR / 'Longformer_berkeley_mimic_GEP(C&O)'),
    'Compliance': str(MODEL_DIR / 'Longformer_berkeley_mimic_GEP(Compliance)'),
    'Descriptors': str(MODEL_DIR / 'Longformer_berkeley_mimic_GEP(Descriptors)'),
    'Misgendering': str(MODEL_DIR / 'Longformer_berkeley_mimic_GEP(Misgendering)'),
}

print("\n Evaluating MSTL Longformer models...")

for subtype, mdir in MSTL_LONGFORMER_DIRS.items():
    # Get subtype labels directly from the test CSV
    labels = df[subtype].astype(int).to_numpy()

    if not os.path.isdir(mdir):
        print(f"[WARN] Missing MSTL model directory: {mdir}")
        continue

    res = eval_transformer_dir(mdir, "Longformer", texts, labels, group_mask=gep, batch_size=48)

    for group_name, mets in res.items():
        row = {
            "Optimization": "MSTL",
            "Family": "Longformer-MSTL",
            "ModelId": os.path.basename(mdir),
            "Subtype": subtype,
            "Group": group_name,
            "N": (int(len(labels)) if group_name == "Overall" else mets.get("N", int(np.sum(gep == int(group_name[-1]))))),
            "Accuracy": mets["Accuracy"],
            "Precision": mets["Precision"],
            "Recall": mets["Recall"],
            "F1": mets["F1"],
            "AUC": mets["AUC"],
        }
        all_rows.append(row)

print(" MSTL Longformer evaluation complete and appended to results.")

# === Save combined output ===
updated_df = pd.DataFrame(all_rows)
updated_csv = os.path.join(RESULTS_DIR, "subtype_best_models_overall_and_GEP_split_with_MSTL.csv")
updated_xlsx = updated_csv.replace(".csv", ".xlsx")

updated_df.to_csv(updated_csv, index=False)
try:
    updated_df.to_excel(updated_xlsx, index=False)
except Exception:
    pass

print(f"\n All results saved (with MSTL):\n - {updated_csv}\n - {updated_xlsx}")
print("\n Preview:")
print(updated_df.tail(12))


In [ ]:
updated_df.shape

In [ ]:
updated_df.head(10)

In [ ]:
out_df.head(20)

In [ ]:
out_df.tail(20)

In [ ]:
for subtype in SUBTYPES:
    percentage = df[subtype].value_counts(normalize=True) * 100
    print(f"Percentage for {subtype}:")
    display(percentage)

In [ ]:
# === Summarize Subtype Performance (Overall + GEP Split) ===
import pandas as pd
import numpy as np

# Load your evaluation file
csv_path = str(RESULTS_DIR / 'subtype_best_models_overall_and_GEP_split_with_MSTL.csv')
df = pd.read_csv(csv_path)

# Ensure sorting order
df = df.sort_values(["Subtype", "Optimization", "Family", "Group"]).reset_index(drop=True)

# Compute mean metrics by group per subtype & optimization
summary = (
    df.groupby(["Optimization", "Subtype", "Group", "Family"], as_index=False)
      .agg({"Accuracy":"mean", "Precision":"mean", "Recall":"mean", "F1":"mean", "AUC":"mean"})
)

# Round for readability
summary[["Accuracy", "Precision", "Recall", "F1", "AUC"]] = summary[
    ["Accuracy", "Precision", "Recall", "F1", "AUC"]
].applymap(lambda x: round(x, 4))

# Filter for “Overall” rows only (main table in paper)
overall_summary = summary[summary["Group"] == "Overall"].drop(columns="Group")

# Save for manuscript / appendix
overall_summary_csv = str(RESULTS_DIR / 'Table8_Subtype_Performance_Overall.csv')
summary_csv = str(RESULTS_DIR / 'Table8_Subtype_Performance_Detailed.csv')
overall_summary.to_csv(overall_summary_csv, index=False)
summary.to_csv(summary_csv, index=False)

print(" Summary tables saved:")
print(f" - Overall results: {overall_summary_csv}")
print(f" - Detailed (with GEP splits): {summary_csv}")
print("\nHead:")
print(overall_summary.head(12))


In [ ]:
# === Pick the single best model per subtype (highest overall F1) ===
import pandas as pd
import numpy as np

# Load the detailed evaluation file
csv_path = str(RESULTS_DIR / 'subtype_best_models_overall_and_GEP_split_with_MSTL.csv')
df = pd.read_csv(csv_path)

# Keep only overall results (not GEP split)
df = df[df["Group"] == "Overall"].copy()

# Select the single best model per subtype by F1 (tie-breaker: Accuracy, then AUC)
best_models_accuracy = (
    df.sort_values(["Subtype", "Accuracy", "F1", "AUC"], ascending=[True, False, False, False])
      .groupby("Subtype", as_index=False)
      .first()
      .sort_values("Subtype")
)

# Round values for readability
for col in ["Accuracy", "Precision", "Recall", "F1", "AUC"]:
    best_models_accuracy[col] = best_models_accuracy[col].round(4)

# Save final summary table
out_csv = str(RESULTS_DIR / 'Table8_Best_Model_Per_Subtype_accuracy.csv')
best_models_accuracy.to_csv(out_csv, index=False)
print(" Saved one-best-model-per-subtype summary:")
print(f" {out_csv}")
print(best_models_accuracy)


In [ ]:
# === Pick the single best model per subtype (highest overall F1) ===
import pandas as pd
import numpy as np

# Load the detailed evaluation file
csv_path = str(RESULTS_DIR / 'subtype_best_models_overall_and_GEP_split_with_MSTL.csv')
df = pd.read_csv(csv_path)

# Keep only overall results (not GEP split)
df = df[df["Group"] == "Overall"].copy()

# Select the single best model per subtype by F1 (tie-breaker: Accuracy, then AUC)
best_models_f1 = (
    df.sort_values(["Subtype", "F1", "Accuracy", "AUC"], ascending=[True, False, False, False])
      .groupby("Subtype", as_index=False)
      .first()
      .sort_values("Subtype")
)

# Round values for readability
for col in ["Accuracy", "Precision", "Recall", "F1", "AUC"]:
    best_models_f1[col] = best_models_f1[col].round(4)

# Save final summary table
out_csv = str(RESULTS_DIR / 'Table8_Best_Model_Per_Subtype_f1.csv')
best_models_f1.to_csv(out_csv, index=False)
print(" Saved one-best-model-per-subtype summary:")
print(f" {out_csv}")
print(best_models_f1)


In [ ]:
best_models_accuracy

In [ ]:
best_models_f1